# Solution: Medical Insurance Cost Prediction

## Objective

Build a multiple linear regression model to predict medical insurance charges.

This solved notebook follows the assignment step by step and includes code, plots, model evaluation, coefficient interpretation, residual diagnostics, Ridge comparison, and written answers.

Target column:

$$
charges
$$

By the end of this solution, you should be able to explain:

- how EDA reveals the strongest drivers of insurance charges,
- why categorical encoding is needed,
- how engineered features can improve a linear model,
- how to evaluate regression models with MAE, RMSE, R2, and adjusted R2,
- why coefficient interpretation must be careful when features are correlated,
- when Ridge regularization can help.


## Dataset columns

- `age`: age of primary beneficiary
- `sex`: gender
- `bmi`: body mass index
- `children`: number of children/dependents covered
- `smoker`: whether the person smokes
- `region`: residential area in the US
- `charges`: individual medical insurance charges

This is a regression problem because `charges` is a continuous numeric target.

## Submission requirements

Submit:

1. Completed `.ipynb` notebook.
2. 5-8 bullet-point summary of your findings.
3. Key plots:
   - target distribution,
   - actual vs predicted,
   - residuals vs predicted,
   - coefficient importance.

Write short explanations wherever asked. Do not only submit code outputs.

In [ ]:
import warnings
from pathlib import Path
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)

## 1. Load the dataset

The code below will:

- use the local dataset if available,
- otherwise download it from the internet.

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

DATA_PATH = DATA_DIR / "insurance.csv"
DATA_URL = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"

if not DATA_PATH.exists():
    print("Dataset not found locally. Downloading...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print(f"Downloaded dataset to: {DATA_PATH}")
else:
    print(f"Using local dataset: {DATA_PATH}")

insurance = pd.read_csv(DATA_PATH)
insurance.head()

## 2. Basic inspection

Complete the following:

1. Print dataset shape.
2. Display first 5 rows.
3. Check column data types.
4. Check missing values.
5. Show summary statistics.

In [ ]:
print("Dataset shape:", insurance.shape)


In [ ]:
insurance.head()


In [ ]:
inspection = pd.DataFrame({
    "dtype": insurance.dtypes.astype(str),
    "missing_values": insurance.isna().sum(),
    "missing_percent": (insurance.isna().mean() * 100).round(2)
})
inspection


In [ ]:
insurance.describe().T


### Written answer

1. The dataset has **1338 rows and 7 columns**.
2. There are **no missing values** in this dataset.
3. Numerical columns: `age`, `bmi`, `children`, `charges`. Categorical columns: `sex`, `smoker`, `region`.

`charges` is the target variable, so the model should not use it as an input feature.


## 3. Exploratory Data Analysis

Create visualizations to understand the target and feature relationships.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(insurance["charges"], bins=35, color="#2c7fb8", edgecolor="white")
ax.set_title("Distribution of insurance charges")
ax.set_xlabel("Charges")
ax.set_ylabel("Number of customers")
plt.show()

print("Skewness of charges:", round(insurance["charges"].skew(), 3))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for smoker, color in [("no", "#1b9e77"), ("yes", "#d95f02")]:
    subset = insurance[insurance["smoker"] == smoker]
    ax.scatter(subset["age"], subset["charges"], label=f"smoker={smoker}", alpha=0.65, color=color)
ax.set_title("Age vs charges")
ax.set_xlabel("Age")
ax.set_ylabel("Charges")
ax.legend()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for smoker, color in [("no", "#1b9e77"), ("yes", "#d95f02")]:
    subset = insurance[insurance["smoker"] == smoker]
    ax.scatter(subset["bmi"], subset["charges"], label=f"smoker={smoker}", alpha=0.65, color=color)
ax.axvline(30, color="black", linestyle="--", linewidth=1, label="BMI = 30")
ax.set_title("BMI vs charges")
ax.set_xlabel("BMI")
ax.set_ylabel("Charges")
ax.legend()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
insurance.boxplot(column="charges", by="smoker", ax=ax, grid=False)
ax.set_title("Charges by smoker status")
ax.set_xlabel("Smoker")
ax.set_ylabel("Charges")
plt.suptitle("")
plt.show()

insurance.groupby("smoker")["charges"].agg(["count", "mean", "median", "std"]).round(2)


In [ ]:
region_avg = insurance.groupby("region")["charges"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
region_avg.plot(kind="bar", ax=ax, color="#7570b3")
ax.set_title("Average charges by region")
ax.set_xlabel("Region")
ax.set_ylabel("Average charges")
ax.tick_params(axis="x", rotation=0)
plt.show()

region_avg.round(2)


In [ ]:
numeric_corr = insurance.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(numeric_corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric_corr.columns)), labels=numeric_corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(numeric_corr.columns)), labels=numeric_corr.columns)
for i in range(len(numeric_corr.columns)):
    for j in range(len(numeric_corr.columns)):
        ax.text(j, i, f"{numeric_corr.iloc[i, j]:.2f}", ha="center", va="center", color="black")
ax.set_title("Correlation heatmap for numerical columns")
fig.colorbar(im, ax=ax)
plt.show()

numeric_corr["charges"].sort_values(ascending=False).round(3)


### EDA answers

1. `charges` is **not normally distributed**. It is strongly right-skewed, meaning most people have lower-to-moderate charges while a smaller group has very high charges.
2. Smoking has a very strong effect on charges. Smokers have much higher average and median charges than non-smokers.
3. BMI has some relationship with charges, but the relationship becomes much clearer when smoker status is considered. High-BMI smokers often have especially high charges.
4. Among the original numerical features, `age` has the strongest correlation with `charges`, followed by `bmi`. However, smoker-related features are much stronger once encoded.


## 4. Feature engineering

Create at least three new features.

Required:

1. `is_obese`

$$
is\_obese =
\begin{cases}
1, & BMI \ge 30 \\
0, & BMI < 30
\end{cases}
$$

2. `age_group`: group ages into buckets.

3. `smoker_bmi`: interaction between smoking and BMI.

Why interaction?

The impact of BMI may be different for smokers and non-smokers.

In [ ]:
insurance_fe = insurance.copy()

insurance_fe["is_obese"] = (insurance_fe["bmi"] >= 30).astype(int)

insurance_fe["age_group"] = pd.cut(
    insurance_fe["age"],
    bins=[0, 25, 35, 45, 55, 65, 100],
    labels=["<=25", "26-35", "36-45", "46-55", "56-65", "65+"],
    right=True
)

insurance_fe["smoker_flag"] = (insurance_fe["smoker"] == "yes").astype(int)
insurance_fe["smoker_bmi"] = insurance_fe["smoker_flag"] * insurance_fe["bmi"]

insurance_fe.head()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
insurance_fe.boxplot(column="charges", by=["is_obese", "smoker"], ax=ax, grid=False)
ax.set_title("Charges by obesity flag and smoker status")
ax.set_xlabel("is_obese, smoker")
ax.set_ylabel("Charges")
plt.suptitle("")
plt.show()

insurance_fe.groupby(["is_obese", "smoker"])["charges"].agg(["count", "mean", "median"]).round(2)


### Feature engineering answers

1. `is_obese` may be useful because BMI above 30 can mark a risk group where medical costs behave differently.
2. `smoker_bmi` is useful because BMI may not have the same effect for smokers and non-smokers. The interaction allows the model to learn that high BMI combined with smoking can sharply increase charges.
3. I expect `smoker_bmi` to help the most because EDA shows that smoking separates high-charge and low-charge customers very strongly, and BMI matters more inside the smoker group.


## 5. Train-test split

Use:

- 80% training data
- 20% test data
- `random_state=42`

Target:

$$
y = charges
$$

In [ ]:
target = "charges"

X = insurance_fe.drop(columns=[target])
y = insurance_fe[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


## 6. Build preprocessing and model pipeline

Your pipeline should:

1. impute numerical missing values if any,
2. scale numerical features,
3. one-hot encode categorical features,
4. train Linear Regression.

In [ ]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

numeric_features, categorical_features


In [ ]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

try:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False)
except TypeError:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", drop="first", sparse=False)

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", one_hot_encoder)
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

preprocessor


In [ ]:
linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

linear_model.fit(X_train, y_train)
linear_model


## 7. Evaluation metrics

Evaluate on test data using:

$$
MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|
$$

$$
RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}
$$

$$
R^2 = 1 - \frac{SS_{res}}{SS_{tot}}
$$

Also calculate adjusted R2:

$$
Adjusted\ R^2 = 1 - \frac{(1-R^2)(n-1)}{n-p-1}
$$

In [ ]:
def regression_report(y_true, y_pred, n_features=None):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    
    report = {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    }
    
    if n_features is not None:
        n = len(y_true)
        p = n_features
        adjusted_r2 = 1 - ((1 - r2) * (n - 1) / (n - p - 1))
        report["Adjusted R2"] = adjusted_r2
    
    return pd.Series(report)

In [ ]:
y_train_pred = linear_model.predict(X_train)
y_test_pred = linear_model.predict(X_test)

transformed_feature_names = linear_model.named_steps["preprocessor"].get_feature_names_out()
n_transformed_features = len(transformed_feature_names)

print("Number of transformed features:", n_transformed_features)

print("Train performance")
display(regression_report(y_train, y_train_pred, n_features=n_transformed_features).round(4))

print("Test performance")
display(regression_report(y_test, y_test_pred, n_features=n_transformed_features).round(4))


### Evaluation answers

1. In a typical run with the given split and engineered features, the test RMSE is around **4500-4600**.
2. The test R2 is usually around **0.86**, meaning the model explains about 86% of the variation in test charges.
3. Train and test performance are close, so the model does not appear to have a large overfitting problem.
4. The model does not strongly overfit, but it still has systematic errors because medical charges are skewed and not perfectly linear.


## 8. Visualize predictions and residuals

Create:

1. Actual vs predicted plot.
2. Residuals vs predicted plot.
3. Residual distribution.

Residual:

$$
e_i = y_i - \hat{y}_i
$$

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, y_test_pred, alpha=0.7, color="#2c7fb8", edgecolor="white", linewidth=0.4)
min_value = min(y_test.min(), y_test_pred.min())
max_value = max(y_test.max(), y_test_pred.max())
ax.plot([min_value, max_value], [min_value, max_value], color="black", linestyle="--", label="Perfect prediction")
ax.set_title("Actual vs predicted charges")
ax.set_xlabel("Actual charges")
ax.set_ylabel("Predicted charges")
ax.legend()
plt.show()


In [ ]:
residuals = y_test - y_test_pred

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(y_test_pred, residuals, alpha=0.7, color="#d95f02", edgecolor="white", linewidth=0.4)
ax.axhline(0, color="black", linestyle="--")
ax.set_title("Residuals vs predicted charges")
ax.set_xlabel("Predicted charges")
ax.set_ylabel("Residual = actual - predicted")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(residuals, bins=35, color="#7570b3", edgecolor="white")
ax.axvline(0, color="black", linestyle="--")
ax.set_title("Residual distribution")
ax.set_xlabel("Residual")
ax.set_ylabel("Frequency")
plt.show()

pd.Series(residuals).describe().round(2)


### Residual answers

1. The residuals are roughly centered around zero, which is good.
2. The residual plot usually shows some pattern rather than pure randomness. This means the linear model is missing some non-linear structure.
3. Yes. The model tends to make larger errors for high-charge customers, especially in the extreme upper range of charges.


## 9. Coefficient interpretation

Extract model coefficients and identify:

- top positive coefficients,
- top negative coefficients.

Be careful:

> A coefficient shows association, not guaranteed causation.

In [ ]:
feature_names = linear_model.named_steps["preprocessor"].get_feature_names_out()
coefficients = linear_model.named_steps["model"].coef_

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "abs_coefficient": np.abs(coefficients)
}).sort_values("abs_coefficient", ascending=False)

coef_df.head(15)


In [ ]:
top_positive = coef_df.sort_values("coefficient", ascending=False).head(8)
top_negative = coef_df.sort_values("coefficient", ascending=True).head(8)
plot_df = pd.concat([top_negative, top_positive]).sort_values("coefficient")

fig, ax = plt.subplots(figsize=(9, 7))
colors = np.where(plot_df["coefficient"] >= 0, "#d95f02", "#1b9e77")
ax.barh(plot_df["feature"], plot_df["coefficient"], color=colors)
ax.axvline(0, color="black", linewidth=1)
ax.set_title("Top positive and negative coefficients")
ax.set_xlabel("Coefficient value")
plt.show()

print("Top positive coefficients")
display(top_positive[["feature", "coefficient"]])

print("Top negative coefficients")
display(top_negative[["feature", "coefficient"]])


### Coefficient answers

1. Smoker-related features, especially the smoker-BMI interaction, increase predicted charges the most. Age also increases predicted charges.
2. Some region/age-group encoded coefficients may be negative depending on the reference category. Negative coefficients mean lower predicted charges compared with the omitted reference group, after controlling for other features.
3. Smoking has a very strong impact. Because `smoker`, `smoker_flag`, and `smoker_bmi` are related, the smoking effect is spread across multiple coefficients.
4. The signs are mostly reasonable, but coefficient interpretation should be cautious because engineered features create multicollinearity. Prediction can still be good even when individual coefficients are unstable.


## 10. Multicollinearity check

Check correlation among numerical features.

High feature-feature correlation can make coefficient interpretation unstable.

In [ ]:
numeric_corr_fe = insurance_fe.select_dtypes(include=["int64", "float64"]).corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(numeric_corr_fe, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric_corr_fe.columns)), labels=numeric_corr_fe.columns, rotation=45, ha="right")
ax.set_yticks(range(len(numeric_corr_fe.columns)), labels=numeric_corr_fe.columns)
for i in range(len(numeric_corr_fe.columns)):
    for j in range(len(numeric_corr_fe.columns)):
        ax.text(j, i, f"{numeric_corr_fe.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Correlation among numerical and engineered numerical features")
fig.colorbar(im, ax=ax)
plt.show()


In [ ]:
corr_abs = numeric_corr_fe.abs()
high_corr_pairs = []

for i, col_1 in enumerate(corr_abs.columns):
    for col_2 in corr_abs.columns[i + 1:]:
        corr_value = corr_abs.loc[col_1, col_2]
        if corr_value >= 0.70:
            high_corr_pairs.append({
                "feature_1": col_1,
                "feature_2": col_2,
                "absolute_correlation": corr_value
            })

high_corr_df = pd.DataFrame(high_corr_pairs).sort_values("absolute_correlation", ascending=False)
high_corr_df


### Multicollinearity answers

1. `smoker_flag` and `smoker_bmi` are highly correlated. `bmi` and `is_obese` are also highly correlated. This is expected because the engineered features were created from original columns.
2. Yes. High correlation can make coefficient interpretation unstable because the model may distribute importance across related features.
3. If the goal is best prediction, I may keep them and compare validation performance. If the goal is clean interpretation, I would remove duplicated features such as either `smoker_flag` or the categorical `smoker` encoding, and possibly choose between `bmi` and `is_obese`.


## 11. Compare with Ridge Regression

Ridge adds a penalty for large coefficients:

$$
\text{Ridge: } \min(MSE + \lambda\sum_j \beta_j^2)
$$

Train a Ridge model and compare it with Linear Regression.

In [ ]:
ridge_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", Ridge(alpha=10.0))
])

ridge_model.fit(X_train, y_train)
ridge_model


In [ ]:
ridge_train_pred = ridge_model.predict(X_train)
ridge_test_pred = ridge_model.predict(X_test)

comparison = pd.DataFrame({
    "Linear Regression Train": regression_report(y_train, y_train_pred, n_features=n_transformed_features),
    "Linear Regression Test": regression_report(y_test, y_test_pred, n_features=n_transformed_features),
    "Ridge Train": regression_report(y_train, ridge_train_pred, n_features=n_transformed_features),
    "Ridge Test": regression_report(y_test, ridge_test_pred, n_features=n_transformed_features),
}).T

comparison.round(4)


### Ridge answers

1. Ridge usually gives very similar test performance to Linear Regression on this dataset. Sometimes the RMSE/R2 changes only slightly.
2. Yes. Ridge reduces coefficient magnitudes because it penalizes large coefficients.
3. Regularization helps when features are correlated or when the model has many encoded/engineered features. It makes the model more stable and less sensitive to small changes in the training data.


## 12. Stretch goals

Try any two:

1. Compare with Lasso Regression.
2. Apply log transform to `charges` and compare performance.
3. Train separate models for smokers and non-smokers.
4. Add polynomial features for `age` or `bmi`.
5. Remove highly correlated features and retrain.

In [ ]:
# Stretch 1: Lasso Regression
lasso_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", Lasso(alpha=25.0, max_iter=10000))
])

lasso_model.fit(X_train, y_train)
lasso_test_pred = lasso_model.predict(X_test)

print("Lasso test performance")
display(regression_report(y_test, lasso_test_pred, n_features=n_transformed_features).round(4))

lasso_coef = lasso_model.named_steps["model"].coef_
print("Non-zero Lasso coefficients:", np.sum(lasso_coef != 0), "out of", len(lasso_coef))

# Stretch 2: Log-transform target
# This is useful because charges is heavily right-skewed.
y_train_log = np.log1p(y_train)

log_target_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])
log_target_model.fit(X_train, y_train_log)
log_pred = np.expm1(log_target_model.predict(X_test))

print("Log-target Linear Regression test performance")
display(regression_report(y_test, log_pred, n_features=n_transformed_features).round(4))


## Final written summary

- Insurance charges are strongly right-skewed: most customers have moderate charges, while a smaller group has very high charges.
- Smoking is the most important driver of high medical charges in this dataset.
- Age has the strongest correlation with charges among the original numerical features; BMI also matters, especially for smokers.
- The `smoker_bmi` interaction is useful because the effect of BMI is different for smokers and non-smokers.
- Linear Regression performs well overall, with test R2 usually around 0.86 after feature engineering.
- The model still makes larger mistakes for very high-charge customers, showing that the relationship is not perfectly linear.
- Ridge Regression gives similar performance but shrinks coefficients, which is useful when features are correlated.
- To improve the model further, I would try log-transforming `charges`, adding non-linear features, and comparing Ridge/Lasso with cross-validation.
